<a href="https://colab.research.google.com/github/Protein-Function-Prediction/COMP3608GroceryPricePrediction/blob/main/ProteinFunctionPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Function Prediction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Dataset 1 : Bioinformatics Simulated

In [ ]:
import zipfile
import pandas as pd

zip_path = "data.zip"

with zipfile.ZipFile("/content/IntelliSysProj/archive.zip", 'r') as z:
    print(z.namelist())  # see all files inside

    with z.open('proteinas_train.csv') as f:
        df1 = pd.read_csv(f)

df1.head()

['proteinas_20000_enriquecido.csv', 'proteinas_test.csv', 'proteinas_train.csv']


,ID_Proteína,Sequência,Massa_Molecular,Ponto_Isoelétrico,Hidrofobicidade,Carga_Total,Proporção_Polar,Proporção_Apolar,Comprimento_Sequência,Classe
0,TRAIN_P00001,GNMRFVLHDEETHWGTLRTTLNCVPSDIYTISGEDSLFWGMAHPFC...,20362.9468,4.866123,0.149425,-3,0.241379,0.408046,174,Estrutural
1,TRAIN_P00002,LFKMQCSFYLLYLAKEAASYQVSMNMLCYEWYNYVYQVTVILRLSR...,9328.7909,6.298636,0.217105,0,0.210526,0.513158,76,Estrutural
2,TRAIN_P00003,PAHLWPYWRFYVWIVFYGYHNPNYHFGMKEVKERPDCKNCTVAVLF...,17616.3852,8.458977,0.192568,8,0.141892,0.466216,148,Estrutural
3,TRAIN_P00004,GEAFSRPHCFACAATKKGFPWARMCCTTSMAMDGVQSKMHKSKHRF...,35244.2968,8.448340,0.160473,21,0.189189,0.408784,296,Estrutural
4,TRAIN_P00005,HYVFQGLMLHCGGYMITACGFGVIFPEQMTREGLIMHTARAHHFLI...,34557.9931,7.696306,0.140411,18,0.202055,0.380137,292,Receptora


Translating the columns and class labels from spanish to english for better readibility.

In [116]:
df1 = df1.rename(columns={
    "ID_Proteína": "Protein_ID",
    "Sequência": "Sequence",
    "Massa_Molecular": "Molecular_Weight",
    "Ponto_Isoelétrico": "Isoelectric_Point",
    "Hidrofobicidade": "Hydrophobicity",
    "Carga_Total": "Net_Charge",
    "Proporção_Polar": "Polar_Ratio",
    "Proporção_Apolar": "NonPolar_Ratio",
    "Comprimento_Sequência": "Sequence_Length",
    "Classe": "Class"
})

In [119]:
df1["Class"] = df1["Class"].replace({
    "Estrutural": "Structural",
    "Receptora": "Receptor",
    "Enzima" : "Enzyme",
    "Transporte" : "Transporter",
    "Outras" : "Others",
})

In [120]:
df1["Class"].head()
df1["Class"].value_counts()

,count
Class,
Enzyme,3235
Structural,3232
Transporter,3225
Others,3183
Receptor,3125


In [124]:
df1.head()

,Protein_ID,Sequence,Molecular_Weight,Isoelectric_Point,Hydrophobicity,Net_Charge,Polar_Ratio,NonPolar_Ratio,Sequence_Length,Class
0,TRAIN_P00001,GNMRFVLHDEETHWGTLRTTLNCVPSDIYTISGEDSLFWGMAHPFC...,20362.9468,4.866123,0.149425,-3,0.241379,0.408046,174,Structural
1,TRAIN_P00002,LFKMQCSFYLLYLAKEAASYQVSMNMLCYEWYNYVYQVTVILRLSR...,9328.7909,6.298636,0.217105,0,0.210526,0.513158,76,Structural
2,TRAIN_P00003,PAHLWPYWRFYVWIVFYGYHNPNYHFGMKEVKERPDCKNCTVAVLF...,17616.3852,8.458977,0.192568,8,0.141892,0.466216,148,Structural
3,TRAIN_P00004,GEAFSRPHCFACAATKKGFPWARMCCTTSMAMDGVQSKMHKSKHRF...,35244.2968,8.448340,0.160473,21,0.189189,0.408784,296,Structural
4,TRAIN_P00005,HYVFQGLMLHCGGYMITACGFGVIFPEQMTREGLIMHTARAHHFLI...,34557.9931,7.696306,0.140411,18,0.202055,0.380137,292,Receptor


# Dataset 2 : CAFA-5-Protein-Function-Prediction

In [ ]:
zip_path2 = "/content/IntelliSysProj/cafa-5-protein-function-prediction.zip"

with zipfile.ZipFile(zip_path2, 'r') as z:
    print(z.namelist())  # see all files inside

    with zipfile.ZipFile(zip_path2, 'r') as z:
      with z.open('Train/train_terms.tsv') as f:
        df2label = pd.read_csv(f, sep="\t")

df2label.head()

['IA.txt', 'Test (Targets)/testsuperset-taxon-list.tsv', 'Test (Targets)/testsuperset.fasta', 'Train/go-basic.obo', 'Train/train_sequences.fasta', 'Train/train_taxonomy.tsv', 'Train/train_terms.tsv', 'sample_submission.tsv']


,EntryID,term,aspect
0,A0A009IHW8,GO:0008152,BPO
1,A0A009IHW8,GO:0034655,BPO
2,A0A009IHW8,GO:0072523,BPO
3,A0A009IHW8,GO:0044270,BPO
4,A0A009IHW8,GO:0006753,BPO


We concatenate the sequence from "train_sequences_fasta" to the existing dataframe joined by the ID.

In [ ]:
sequences = []

with zipfile.ZipFile(zip_path2, 'r') as z:
    with z.open('Train/train_sequences.fasta') as f:
        current_id = None
        current_seq = []

        for line in f:
            line = line.decode('utf-8').strip()

            if line.startswith(">"):
                if current_id is not None:
                    sequences.append((current_id, ''.join(current_seq)))

                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)

        # last entry
        if current_id is not None:
            sequences.append((current_id, ''.join(current_seq)))

In [ ]:
df_seq = pd.DataFrame(sequences, columns=["EntryID", "sequence"])
df_seq.head()

,EntryID,sequence
0,P20536,MNSVTVSHAPYTITYHDDWEPVMSQLVEFYNEVASWLLRDETSPIP...
1,O73864,MTEYRNFLLLFITSLSVIYPCTGISWLGLTINGSSVGWNQTHHCKL...
2,O95231,MRLSSSPPRGPQQLSSFGSVDWLSQSSCSGPTHTPRPADFSLGSLP...
3,A0A0B4J1F4,MGGEAGADGPRGRVKSLGLVFEDESKGCYSSGETVAGHVLLEAAEP...
4,P54366,MVETNSPPAGYTLKRSPSDLGEQQQPPRQISRSPGNTAAYHLTTAM...


Then we merge the both dataframes together.

In [ ]:
df2 = pd.merge(df_seq, df2label, on='EntryID', how='inner')
df2.head()

,EntryID,sequence,term,aspect
0,P20536,MNSVTVSHAPYTITYHDDWEPVMSQLVEFYNEVASWLLRDETSPIP...,GO:0008152,BPO
1,P20536,MNSVTVSHAPYTITYHDDWEPVMSQLVEFYNEVASWLLRDETSPIP...,GO:0071897,BPO
2,P20536,MNSVTVSHAPYTITYHDDWEPVMSQLVEFYNEVASWLLRDETSPIP...,GO:0044249,BPO
3,P20536,MNSVTVSHAPYTITYHDDWEPVMSQLVEFYNEVASWLLRDETSPIP...,GO:0006259,BPO
4,P20536,MNSVTVSHAPYTITYHDDWEPVMSQLVEFYNEVASWLLRDETSPIP...,GO:0009059,BPO


Below are functions used for feature engineering of physiochemical properties derived from the sequence.

### Amino Acid Composition

In [ ]:
from collections import Counter

AA = "ACDEFGHIKLMNPQRSTVWY"

def aa_composition(seq):
    counts = Counter(seq)
    total = len(seq)
    return {f"aa_{aa}": counts.get(aa, 0) / total for aa in AA}

In [ ]:
aa_features = df_seq["sequence"].apply(aa_composition).apply(pd.Series)

### Charge-based features

In [ ]:
def charge_features(seq):
    positive = sum(seq.count(aa) for aa in "KRH")
    negative = sum(seq.count(aa) for aa in "DE")
    total = len(seq)

    return {
        "pos_charge_ratio": positive / total,
        "neg_charge_ratio": negative / total,
        "net_charge": (positive - negative) / total
    }

### Hydrophobicity

In [ ]:
hydro = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5,
    'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4,
    'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9,
    'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8,
    'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

def hydrophobicity(seq):
    return sum(hydro.get(aa, 0) for aa in seq) / len(seq)

### Aromaticity + aliphatic index

In [ ]:
def physico_extra(seq):
    aromatic = sum(seq.count(aa) for aa in "FYW")
    aliphatic = sum(seq.count(aa) for aa in "ILV")

    return {
        "aromaticity": aromatic / len(seq),
        "aliphatic_index": aliphatic / len(seq)
    }

In [ ]:
def build_physicochem_features(seq):
    features = {}
    features.update(aa_composition(seq))
    features.update(charge_features(seq))
    features["hydrophobicity"] = hydrophobicity(seq)
    features.update(physico_extra(seq))
    return features

In [ ]:
physico_df = df_seq["sequence"].apply(build_physicochem_features).apply(pd.Series)

In [ ]:
X_physico = physico_df.fillna(0)

In [ ]:
df2 = pd.merge(X_physico, df2label, left_index=True, right_index=True)
df2.head()

,aa_A,aa_C,aa_D,aa_E,aa_F,aa_G,aa_H,aa_I,aa_K,aa_L,...,aa_Y,pos_charge_ratio,neg_charge_ratio,net_charge,hydrophobicity,aromaticity,aliphatic_index,EntryID,term,aspect
0,0.036697,0.018349,0.059633,0.050459,0.041284,0.045872,0.027523,0.091743,0.077982,0.082569,...,0.055046,0.142202,0.110092,0.032110,-0.242661,0.119266,0.247706,A0A009IHW8,GO:0008152,BPO
1,0.073446,0.064972,0.039548,0.039548,0.022599,0.070621,0.022599,0.031073,0.048023,0.087571,...,0.033898,0.138418,0.079096,0.059322,-0.240113,0.076271,0.183616,A0A009IHW8,GO:0034655,BPO
2,0.093023,0.023256,0.015504,0.038760,0.034884,0.089147,0.027132,0.007752,0.023256,0.108527,...,0.007752,0.116279,0.054264,0.062016,-0.438760,0.058140,0.143411,A0A009IHW8,GO:0072523,BPO
3,0.079518,0.028916,0.031325,0.072289,0.033735,0.086747,0.019277,0.043373,0.033735,0.089157,...,0.033735,0.106024,0.103614,0.002410,-0.114217,0.077108,0.214458,A0A009IHW8,GO:0044270,BPO
4,0.103614,0.007229,0.031325,0.050602,0.016867,0.091566,0.074699,0.021687,0.033735,0.081928,...,0.021687,0.161446,0.081928,0.079518,-0.774940,0.043373,0.142169,A0A009IHW8,GO:0006753,BPO


We proceed to make the dataset of a multi-classification type.

In [87]:
df2_grouped = df2.groupby("EntryID").agg({
    "term": list,   # GO terms
    # keep ONE copy of features (they're identical anyway)
    "aa_A": "first",
    "aa_C": "first",
    'aa_D' : "first",
    'aa_E' : "first",
    'aa_F' : "first", 'aa_G': "first", 'aa_H': "first", 'aa_I': "first", 'aa_K': "first",
       'aa_L': "first", 'aa_M': "first", 'aa_N' : "first", 'aa_P': "first", 'aa_Q': "first", 'aa_R': "first", 'aa_S': "first", 'aa_T': "first", 'aa_V': "first",
       'aa_W': "first", 'aa_Y': "first", 'pos_charge_ratio': "first", 'neg_charge_ratio': "first", 'net_charge': "first",
       'hydrophobicity': "first", 'aromaticity': "first", 'aliphatic_index': "first"
}).reset_index()

In [88]:
from collections import Counter

def pick_label(go_list):
    return Counter(go_list).most_common(1)[0][0]

df2_grouped["GO_label"] = df2_grouped["term"].apply(pick_label)

In [90]:
df2 = df2_grouped.drop(columns=["term"])

In [91]:
df2.head()

,EntryID,aa_A,aa_C,aa_D,aa_E,aa_F,aa_G,aa_H,aa_I,aa_K,...,aa_V,aa_W,aa_Y,pos_charge_ratio,neg_charge_ratio,net_charge,hydrophobicity,aromaticity,aliphatic_index,GO_label
0,A0A009IHW8,0.036697,0.018349,0.059633,0.050459,0.041284,0.045872,0.027523,0.091743,0.077982,...,0.073394,0.022936,0.055046,0.142202,0.110092,0.032110,-0.242661,0.119266,0.247706,GO:0008152
1,A0A021WW32,0.154440,0.015444,0.077220,0.061776,0.030888,0.069498,0.015444,0.073359,0.073359,...,0.061776,0.003861,0.023166,0.135135,0.138996,-0.003861,-0.013514,0.057915,0.220077,GO:0048869
2,A0A023FFD0,0.073593,0.020202,0.044733,0.076479,0.030303,0.073593,0.023088,0.033189,0.056277,...,0.067821,0.011544,0.030303,0.151515,0.121212,0.030303,-0.395960,0.072150,0.219336,GO:0060302
3,A0A023GPJ3,0.036855,0.041769,0.034398,0.090909,0.049140,0.046683,0.054054,0.049140,0.105651,...,0.034398,0.004914,0.041769,0.208845,0.125307,0.083538,-0.844963,0.095823,0.159705,GO:0046888
4,A0A023GPK8,0.066225,0.006623,0.072848,0.052980,0.019868,0.079470,0.019868,0.086093,0.092715,...,0.105960,0.000000,0.026490,0.132450,0.125828,0.006623,0.075497,0.046358,0.291391,GO:0090596


In [94]:
print(df2.columns)
print(df2.head())

Index(['EntryID', 'aa_A', 'aa_C', 'aa_D', 'aa_E', 'aa_F', 'aa_G', 'aa_H',
       'aa_I', 'aa_K', 'aa_L', 'aa_M', 'aa_N', 'aa_P', 'aa_Q', 'aa_R', 'aa_S',
       'aa_T', 'aa_V', 'aa_W', 'aa_Y', 'pos_charge_ratio', 'neg_charge_ratio',
       'net_charge', 'hydrophobicity', 'aromaticity', 'aliphatic_index',
       'GO_label'],
      dtype='object')
      EntryID      aa_A      aa_C      aa_D      aa_E      aa_F      aa_G  \
0  A0A009IHW8  0.036697  0.018349  0.059633  0.050459  0.041284  0.045872   
1  A0A021WW32  0.154440  0.015444  0.077220  0.061776  0.030888  0.069498   
2  A0A023FFD0  0.073593  0.020202  0.044733  0.076479  0.030303  0.073593   
3  A0A023GPJ3  0.036855  0.041769  0.034398  0.090909  0.049140  0.046683   
4  A0A023GPK8  0.066225  0.006623  0.072848  0.052980  0.019868  0.079470   

       aa_H      aa_I      aa_K  ...      aa_V      aa_W      aa_Y  \
0  0.027523  0.091743  0.077982  ...  0.073394  0.022936  0.055046   
1  0.015444  0.073359  0.073359  ...  0.061776  0.

# Dataset 3 : Protein sequences w GO Annotations

In [ ]:
zip_path3 = "/content/IntelliSysProj/archive (1).zip"

with zipfile.ZipFile(zip_path3, 'r') as z:
    print(z.namelist())  # see all files inside

    with zipfile.ZipFile(zip_path3, 'r') as z:
      with z.open('proteins.csv') as f:
        df3 = pd.read_csv(f)

df3.head()

['proteins.csv']


,Entry,Sequence,GO_list,GO,GO_id,GO_namespace,GO_parents,GO_children,seq_length,mol_weight,...,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
0,C0SPC1,MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...,"['cytoplasm [GO:0005737]', 'ATP binding [GO:00...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",269.0,30790.1862,...,12.0,11.0,11.0,10.0,9.0,14.0,8.0,14.0,10.0,8.0
1,O05512,MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...,"['extracellular region [GO:0005576]', 'mannan ...",extracellular region [GO:0005576],GO:0005576,cellular_component,"('GO:0110165',)","('GO:0043083', 'GO:0099544', 'GO:0098595', 'GO...",362.0,40891.5038,...,7.0,24.0,16.0,14.0,10.0,27.0,22.0,15.0,11.0,20.0
2,O06724,MKFATGELYNRMFVGLIIDDEKIMDLQKAEKKLFELETIPGSLIEC...,"['cytoplasm [GO:0005737]', 'acetylpyruvate hyd...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",301.0,33145.5860,...,9.0,7.0,16.0,8.0,10.0,22.0,19.0,17.0,1.0,6.0
3,O08394,MKETSPIPQPKTFGPLGNLPLIDKDKPTLSLIKLAEEQGPIFQIHT...,"['cytosol [GO:0005829]', 'aromatase activity [...",cytosol [GO:0005829],GO:0005829,cellular_component,"('GO:0110165',)","('GO:0099522',)",1061.0,119466.8051,...,23.0,30.0,56.0,50.0,71.0,59.0,58.0,62.0,12.0,32.0
4,O31616,MKRHYEAVVIGGGIIGSAIAYYLAKENKNTALFESGTMGGRTTSAA...,"['cytoplasm [GO:0005737]', 'FAD binding [GO:00...",cytoplasm [GO:0005737],GO:0005737,cellular_component,"('GO:0110165',)","('GO:1990917', 'GO:0099568', 'GO:0016528', 'GO...",369.0,40936.3550,...,13.0,11.0,14.0,9.0,16.0,20.0,13.0,26.0,7.0,12.0


In [ ]:
print(df3.columns)
print(df3.head())

Index(['Entry', 'Sequence', 'GO_list', 'GO', 'GO_id', 'GO_namespace',
       'GO_parents', 'GO_children', 'seq_length', 'mol_weight', 'pI', 'gravy',
       'instability', 'aromaticity', 'helix', 'turn', 'sheet', 'aa_A', 'aa_C',
       'aa_D', 'aa_E', 'aa_F', 'aa_G', 'aa_H', 'aa_I', 'aa_K', 'aa_L', 'aa_M',
       'aa_N', 'aa_P', 'aa_Q', 'aa_R', 'aa_S', 'aa_T', 'aa_V', 'aa_W', 'aa_Y'],
      dtype='object')
    Entry                                           Sequence  \
0  C0SPC1  MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...   
1  O05512  MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...   
2  O06724  MKFATGELYNRMFVGLIIDDEKIMDLQKAEKKLFELETIPGSLIEC...   
3  O08394  MKETSPIPQPKTFGPLGNLPLIDKDKPTLSLIKLAEEQGPIFQIHT...   
4  O31616  MKRHYEAVVIGGGIIGSAIAYYLAKENKNTALFESGTMGGRTTSAA...   

                                             GO_list  \
0  ['cytoplasm [GO:0005737]', 'ATP binding [GO:00...   
1  ['extracellular region [GO:0005576]', 'mannan ...   
2  ['cytoplasm [GO:0005737]', 'acetylp

We drop some of the unnecessary columns :

In [115]:
df3cleaned = df3.drop(columns=[
    "GO_namespace",
    "GO_parents",
    "GO_children",
    "GO_list",
])

df3cleaned.head(3000)

,Entry,Sequence,GO,GO_id,seq_length,mol_weight,pI,gravy,instability,aromaticity,...,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
0,C0SPC1,MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...,cytoplasm [GO:0005737],GO:0005737,269.0,30790.1862,5.835909,-0.238290,49.504461,0.092937,...,12.0,11.0,11.0,10.0,9.0,14.0,8.0,14.0,10.0,8.0
1,O05512,MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...,extracellular region [GO:0005576],GO:0005576,362.0,40891.5038,5.810275,-0.353039,33.137597,0.129834,...,7.0,24.0,16.0,14.0,10.0,27.0,22.0,15.0,11.0,20.0
2,O06724,MKFATGELYNRMFVGLIIDDEKIMDLQKAEKKLFELETIPGSLIEC...,cytoplasm [GO:0005737],GO:0005737,301.0,33145.5860,5.545519,-0.268771,28.630897,0.063123,...,9.0,7.0,16.0,8.0,10.0,22.0,19.0,17.0,1.0,6.0
3,O08394,MKETSPIPQPKTFGPLGNLPLIDKDKPTLSLIKLAEEQGPIFQIHT...,cytosol [GO:0005829],GO:0005829,1061.0,119466.8051,5.786403,-0.474458,31.796230,0.081056,...,23.0,30.0,56.0,50.0,71.0,59.0,58.0,62.0,12.0,32.0
4,O31616,MKRHYEAVVIGGGIIGSAIAYYLAKENKNTALFESGTMGGRTTSAA...,cytoplasm [GO:0005737],GO:0005737,369.0,40936.3550,5.919520,-0.173984,35.713008,0.100271,...,13.0,11.0,14.0,9.0,16.0,20.0,13.0,26.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,P0A8G6,MAKVLVLYYSMYGHIETMARAVAEGASKVDGAEVVVKRVPETMPPQ...,cytosol [GO:0005829],GO:0005829,322.0,35042.2922,5.491522,0.090994,29.750621,0.059006,...,11.0,8.0,13.0,13.0,16.0,12.0,19.0,22.0,5.0,8.0
2996,P0A8H6,MKPSSSNSRSKGHAKARRKTREELDQEARDRKRQKKRRGHAPGSRA...,cytosol [GO:0005829],GO:0005829,173.0,17680.7756,5.946745,-0.276879,35.371098,0.023121,...,4.0,5.0,10.0,6.0,3.0,11.0,8.0,12.0,0.0,1.0
2997,P0A8M3,MPVITLPDGSQRHYDHAVSPMDVALDIGPGLAKACIAGRVNGELVD...,cytoplasm [GO:0005737],GO:0005737,188.0,20375.3260,5.006458,0.059574,35.433511,0.058511,...,7.0,6.0,12.0,14.0,7.0,9.0,9.0,18.0,1.0,4.0
2998,P0A8N7,MSETASWQPSASIPNLLKRAAIMAEIRRFFADRGVLEVETPCMSQA...,cytoplasm [GO:0005737],GO:0005737,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [104]:
df3cleaned.count()

,0
Entry,81000
Sequence,81000
GO,81000
GO_id,81000
seq_length,65205
mol_weight,65205
pI,65205
gravy,65205
instability,65205
aromaticity,65205


Biopython was installed and used to extract protein physicochemical properties from amino acid sequences, enabling the construction of structured numerical feature vectors.

In [107]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.2 MB/s eta 0:00:00


In [110]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

AA_LIST = "ACDEFGHIKLMNPQRSTVWY"

def compute_features(seq):
    seq = str(seq).upper()

    # --- replace uncommon amino acids ---
    seq = seq.replace("U", "X")  # selenocysteine
    seq = seq.replace("B", "X")  # D or N
    seq = seq.replace("Z", "X")  # E or Q
    seq = seq.replace("J", "X")  # rare ambiguous

    if len(seq) == 0:
        return None

    pa = ProteinAnalysis(seq)

    aa_counts = {
        f"aa_{aa}": seq.count(aa) / len(seq)
        for aa in AA_LIST
    }

    helix, turn, sheet = pa.secondary_structure_fraction()

    return {
        "seq_length": len(seq),
        "mol_weight": pa.molecular_weight(),
        "pI": pa.isoelectric_point(),
        "gravy": pa.gravy(),
        "instability": pa.instability_index(),
        "aromaticity": pa.aromaticity(),
        "helix": helix,
        "turn": turn,
        "sheet": sheet,
        **aa_counts
    }

In [112]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

AA_LIST = "ACDEFGHIKLMNPQRSTVWY"

def compute_features(seq):
    seq = str(seq).upper()

    # replace ambiguous residues
    seq = seq.replace("U", "X")
    seq = seq.replace("B", "X")
    seq = seq.replace("Z", "X")
    seq = seq.replace("J", "X")

    # for molecular weight ONLY
    clean_seq = seq.replace("X", "")

    if len(clean_seq) == 0:
        return None

    pa = ProteinAnalysis(clean_seq)

    aa_counts = {
        f"aa_{aa}": seq.count(aa) / len(seq)
        for aa in AA_LIST
    }

    helix, turn, sheet = pa.secondary_structure_fraction()

    return {
        "seq_length": len(seq),
        "mol_weight": pa.molecular_weight(),
        "pI": pa.isoelectric_point(),
        "gravy": pa.gravy(),
        "instability": pa.instability_index(),
        "aromaticity": pa.aromaticity(),
        "helix": helix,
        "turn": turn,
        "sheet": sheet,
        **aa_counts
    }

In [113]:
features_df3cleaned = df3cleaned["Sequence"].apply(compute_features).apply(pd.Series)

df3cleaned = pd.concat([df3cleaned[["Entry", "Sequence", "GO", "GO_id"]], features_df3cleaned], axis=1)

df3cleaned.head()

,Entry,Sequence,GO,GO_id,seq_length,mol_weight,pI,gravy,instability,aromaticity,...,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
0,C0SPC1,MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...,cytoplasm [GO:0005737],GO:0005737,269.0,30790.1862,5.835909,-0.238290,49.504461,0.092937,...,0.044610,0.040892,0.040892,0.037175,0.033457,0.052045,0.029740,0.052045,0.037175,0.029740
1,O05512,MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...,extracellular region [GO:0005576],GO:0005576,362.0,40891.5038,5.810275,-0.353039,33.137597,0.129834,...,0.019337,0.066298,0.044199,0.038674,0.027624,0.074586,0.060773,0.041436,0.030387,0.055249
2,O06724,MKFATGELYNRMFVGLIIDDEKIMDLQKAEKKLFELETIPGSLIEC...,cytoplasm [GO:0005737],GO:0005737,301.0,33145.5860,5.545519,-0.268771,28.630897,0.063123,...,0.029900,0.023256,0.053156,0.026578,0.033223,0.073090,0.063123,0.056478,0.003322,0.019934
3,O08394,MKETSPIPQPKTFGPLGNLPLIDKDKPTLSLIKLAEEQGPIFQIHT...,cytosol [GO:0005829],GO:0005829,1061.0,119466.8051,5.786403,-0.474458,31.796230,0.081056,...,0.021678,0.028275,0.052780,0.047125,0.066918,0.055608,0.054665,0.058435,0.011310,0.030160
4,O31616,MKRHYEAVVIGGGIIGSAIAYYLAKENKNTALFESGTMGGRTTSAA...,cytoplasm [GO:0005737],GO:0005737,369.0,40936.3550,5.919520,-0.173984,35.713008,0.100271,...,0.035230,0.029810,0.037940,0.024390,0.043360,0.054201,0.035230,0.070461,0.018970,0.032520


A single unified feature representation was constructed for each dataset by combining physicochemical properties and amino acid composition, ensuring that the same input space could be consistently used across Logistic Regression, SVM, and CNN models for fair performance comparison.

In [125]:
num1 = [
    "Molecular_Weight",
    "Isoelectric_Point",
    "Hydrophobicity",
    "Net_Charge",
    "Polar_Ratio",
    "NonPolar_Ratio",
    "Sequence_Length"
]

from collections import Counter
import numpy as np

AA = list("ACDEFGHIKLMNPQRSTVWY")

def aa_comp(seq):
    c = Counter(seq)
    L = len(seq)
    return np.array([c.get(a, 0)/L for a in AA])


X1_num = df1[num1].values
X1_seq = np.vstack(df1["Sequence"].apply(aa_comp))

X1 = np.hstack([X1_num, X1_seq])

y1 = df1["Class"]

In [127]:
y2 = df2["GO_label"]
X2 = df2.drop(columns=["GO_label", "EntryID"])

X2 = X2.values

In [128]:
num3 = [
    "seq_length",
    "mol_weight",
    "pI",
    "gravy",
    "instability",
    "aromaticity",
    "helix",
    "turn",
    "sheet"
]

aa_cols3 = [
    "aa_A","aa_C","aa_D","aa_E","aa_F","aa_G","aa_H","aa_I","aa_K","aa_L",
    "aa_M","aa_N","aa_P","aa_Q","aa_R","aa_S","aa_T","aa_V","aa_W","aa_Y"
]

X3 = df3[num3 + aa_cols3].values

y3 = df3["GO_id"]

A Conv1D layer expects 3D input, because it is designed to slide a filter over a sequence-like structure.

The feature vectors were reshaped into a 3D tensor format to enable 1D convolution over the feature axis, allowing the model to learn local interactions between adjacent biochemical descriptors.

In [131]:
X1_cnn = X1.reshape(X1.shape[0], X1.shape[1], 1)
X2_cnn = X2.reshape(X2.shape[0], X2.shape[1], 1)
X3_cnn = X3.reshape(X3.shape[0], X3.shape[1], 1)

In [129]:
print(X1.shape)
print(X2.shape)
print(X3.shape)

(16000, 27)
(4502, 26)
(81000, 29)


In [132]:
print(X1_cnn.shape)
print(X2_cnn.shape)
print(X3_cnn.shape)

(16000, 27, 1)
(4502, 26, 1)
(81000, 29, 1)
